In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
1
NVIDIA GeForce RTX 5060 Ti


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# Dataset: https://www.kaggle.com/datasets/nguynquangchiu/mt-en-vi
# Dummy dataset
pairs = [
    ("i love you", "tôi yêu bạn"),
    ("i eat bread", "tôi ăn bánh mì"),
    ("he is tall", "anh ấy cao"),
    ("she is very nice", "cô ấy rất tốt"),
    ("they really like you", "họ rất thích bạn")
]

In [4]:
# 1. Build vocabulary
import torch
import torch.nn as nn
import torch.optim as optim

PAD, SOS, EOS = "<pad>", "<sos>", "<eos>"

def build_vocab(sentences):
    vocab = set()
    for s in sentences:
        vocab.update(s.split())
    vocab = [PAD, SOS, EOS] + sorted(vocab)

    w2i = {w:i for i,w in enumerate(vocab)}
    i2w = {i:w for w,i in w2i.items()}
    return w2i, i2w

eng_sentences = [p[0] for p in pairs]
vie_sentences = [p[1] for p in pairs]

eng_w2i, eng_i2w = build_vocab(eng_sentences)
vie_w2i, vie_i2w = build_vocab(vie_sentences)

print('Source vocabulary size:', len(eng_w2i), eng_w2i)
print('Target vocabulary size:', len(vie_w2i), vie_w2i)

Source vocabulary size: 17 {'<pad>': 0, '<sos>': 1, '<eos>': 2, 'bread': 3, 'eat': 4, 'he': 5, 'i': 6, 'is': 7, 'like': 8, 'love': 9, 'nice': 10, 'really': 11, 'she': 12, 'tall': 13, 'they': 14, 'very': 15, 'you': 16}
Target vocabulary size: 17 {'<pad>': 0, '<sos>': 1, '<eos>': 2, 'anh': 3, 'bánh': 4, 'bạn': 5, 'cao': 6, 'cô': 7, 'họ': 8, 'mì': 9, 'rất': 10, 'thích': 11, 'tôi': 12, 'tốt': 13, 'yêu': 14, 'ăn': 15, 'ấy': 16}


In [5]:
# 2. Encode + pad sentence
def encode(sentence, vocab):
    return [vocab[w] for w in sentence.split()]

def pad(seq, max_len, pad_idx):
    return seq + [pad_idx] * (max_len - len(seq))


data = []
max_src_len = max(len(p[0].split()) for p in pairs)
max_trg_len = max(len(p[1].split()) for p in pairs) + 2  # SOS + EOS

for eng, vie in pairs:
    src = encode(eng, eng_w2i)
    trg = [vie_w2i[SOS]] + encode(vie, vie_w2i) + [vie_w2i[EOS]]

    src = pad(src, max_src_len, eng_w2i[PAD])
    trg = pad(trg, max_trg_len, vie_w2i[PAD])

    data.append((src, trg))

In [6]:
# 3. Encoder-decoder model
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)  # (batch, 1)
        embedded = self.embedding(input)

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))

        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):  # This function used only in training, not inference
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size)
        
        hidden, cell = self.encoder(src)
        input = trg[:, 0]  # <sos>

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t] = output
            input = trg[:, t]  # teacher forcing

        return outputs

In [7]:
# 4. Training model
INPUT_DIM = len(eng_w2i)
OUTPUT_DIM = len(vie_w2i)

model = Seq2Seq(
    Encoder(INPUT_DIM, 32, 64),
    Decoder(OUTPUT_DIM, 32, 64)
)

criterion = nn.CrossEntropyLoss(ignore_index=vie_w2i["<pad>"])
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(500):
    total_loss = 0

    for src, trg in data:
        src = torch.tensor(src).unsqueeze(0)
        trg = torch.tensor(trg).unsqueeze(0)

        output = model(src, trg)

        output = output[:, 1:].reshape(-1, OUTPUT_DIM)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 14.2203
Epoch 50, Loss: 0.0067
Epoch 100, Loss: 0.0025
Epoch 150, Loss: 0.0013
Epoch 200, Loss: 0.0008
Epoch 250, Loss: 0.0006
Epoch 300, Loss: 0.0004
Epoch 350, Loss: 0.0003
Epoch 400, Loss: 0.0002
Epoch 450, Loss: 0.0002


In [8]:
# 5. Inference
import torch
import torch.nn.functional as F

def translate(model, sentence, max_len=50, stop_tokens=None,  
              repetition_window=3,       # detect repeated tokens for stopping trigger
              min_prob_threshold=None,    # optional confidence stopping
              verbose=False):
    model.eval()

    # Encode source
    src = torch.tensor(encode(sentence, eng_w2i)).unsqueeze(0)

    with torch.no_grad():
        hidden, cell = model.encoder(src)

    input = torch.tensor([vie_w2i["<sos>"]])
    outputs = []

    recent_tokens = []

    for step in range(max_len):  # still keep as safety cap
        with torch.no_grad():
            output, hidden, cell = model.decoder(input, hidden, cell)

        probs = F.softmax(output, dim=1)
        pred_token = probs.argmax(1).item()
        pred_prob = probs.max().item()

        # ✅ 1. EOS stopping
        if pred_token == vie_w2i["<eos>"]:
            if verbose:
                print("Stopped: EOS")
            break

        # ✅ 2. Confidence stopping (optional)
        if min_prob_threshold is not None and pred_prob < min_prob_threshold:
            if verbose:
                print("Stopped: low confidence")
            break

        outputs.append(pred_token)

        # ✅ 3. Stop token condition
        if stop_tokens:
            word = vie_i2w[pred_token]
            if word in stop_tokens:
                print("Stopped: stop token")
                break

        # ✅ 4. Repetition detection
        recent_tokens.append(pred_token)
        if len(recent_tokens) > repetition_window:
            recent_tokens.pop(0)

        if len(set(recent_tokens)) == 1 and len(recent_tokens) == repetition_window:
            print("Stopped: repetition loop")
            break

        # Next input
        input = torch.tensor([pred_token])

    return " ".join([vie_i2w[i] for i in outputs])

for pair in pairs:
    print(f'Input: {pair[0]}\t Output: {translate(model, pair[0])}\t Expected: {pair[1]}')


Input: i love you	 Output: tôi yêu bạn	 Expected: tôi yêu bạn
Input: i eat bread	 Output: tôi ăn bánh mì	 Expected: tôi ăn bánh mì
Input: he is tall	 Output: anh ấy cao	 Expected: anh ấy cao
Input: she is very nice	 Output: cô ấy rất tốt	 Expected: cô ấy rất tốt
Input: they really like you	 Output: họ rất thích bạn	 Expected: họ rất thích bạn
